In [ ]:
#Librerias empleadas

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Leer CSVs
ruta_facil = "historial_sudoku0.csv"
ruta_dificil = "historial_sudoku1.csv"

df = pd.read_csv(ruta_facil) # Dependiendo del que quiera analizar pondré uno u otro

df.head()

In [ ]:
# Nos quedamos con: mutacion, ejecucion, fitness_final
df_ordenado = df.sort_values(["mutacion", "ejecucion", "generacion"])

final_por_ejecucion = df_ordenado.groupby(["mutacion", "ejecucion"], as_index=False).tail(1)

final_por_ejecucion = final_por_ejecucion[["mutacion", "ejecucion", "fitness"]].copy()
final_por_ejecucion = final_por_ejecucion.rename(columns={"fitness": "fitness_final"})

final_por_ejecucion.head()

Como no llego a un TE suficiente, no merece la pena graficar ni TE vs Mutación, ni generación de éxito por ejecución ni la gráfica de eficiencia. Vamos con VAMM vs Mutación y la curva de progreso.

In [ ]:
# VAMM (media fitness_final) + dispersión vs mutación
tabla_vamm = final_por_ejecucion.groupby("mutacion")["fitness_final"].agg(["mean", "std", "count"]).reset_index()
tabla_vamm = tabla_vamm.rename(columns={"mean": "VAMM", "std": "desviacion", "count": "n"})

plt.figure()
plt.errorbar(
    tabla_vamm["mutacion"],
    tabla_vamm["VAMM"],
    yerr=tabla_vamm["desviacion"],
    fmt="o-",
    capsize=4
)
plt.xlabel("Probabilidad de mutación (Pm)")
plt.ylabel("VAMM = media del fitness final (± desviación)")
plt.title("VAMM vs mutación (con dispersión)")
plt.xscale("log")  # Adecuamos en "log" porque se verá mejor devido a mis valoes de mutación
plt.grid(True)
plt.show()

tabla_vamm # Buscamos ver la calidad final típica que consigue el algoritmo con x mutación

In [ ]:
#Curva progreso de mutación (fitness vs generaciones) 
plt.figure(figsize=(10, 6))

for pm in sorted(df["mutacion"].unique()):
    
    df_pm = df[df["mutacion"] == pm].copy()
    df_pm = df_pm.sort_values(["ejecucion", "generacion"])
    
    # Tabla: filas = generación, columnas = ejecución
    tabla = df_pm.pivot_table(
        index="generacion",
        columns="ejecucion",
        values="fitness",
        aggfunc="last"
    )

    
    media = tabla.mean(axis=1)

    
    plt.plot(
        media.index,
        media.values,
        label=f"Pm = {pm}",
        linewidth=2,
        alpha=0.8
    )

plt.xlabel("Generación")
plt.ylabel("Fitness medio")
plt.title("Curvas de progreso medias para distintas probabilidades de mutación")
plt.legend(title="Probabilidad de mutación")
plt.grid(True)
plt.show()